### Building the DCT-Based CNN Classifier

After extracting and inspecting the DCT coefficients, we build a Convolutional Neural Network (CNN) to classify images directly from their JPEG-domain representations without first reconstructing the full RGB images.


The proposed model has two branches:

- **Y branch:** Processes the luminance DCT coefficients using one input channel.
- **Chroma branch:** Processes the Cb and Cr DCT coefficients together using two input channels.

Each branch extracts features through convolutional and pooling layers. Adaptive average pooling converts the extracted feature maps into fixed-size feature vectors, allowing the two branches to be combined even when their spatial dimensions differ.

Finally, the model concatenates the two feature vectors and passes them to a fully connected classifier that predicts the image class.

This architecture provides a CNN baseline for evaluating whether image classification can be performed effectively using JPEG DCT coefficients directly. Its performance can later be compared with a conventional RGB-based classifier using metrics such as classification accuracy and inference time.


In [ ]:

import torch
import torch.nn as nn

class DCTCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Branch for luminance coefficients
        self.y_branch = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        # Branch for chrominance coefficients
        self.chroma_branch = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        # Combine the two branches
        self.classifier = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, y, chroma):
        y_features = self.y_branch(y)
        chroma_features = self.chroma_branch(chroma)

        features = torch.cat(
            [y_features, chroma_features],
            dim=1
        )

        return self.classifier(features)